In [1]:
from nd_aligner.config.ndaligner.data_config import DataConfig
from nd_aligner.data.datafactory import DataFactory

data_config = DataConfig()


print("📦 Initializing datasets...")
data_factory = DataFactory(data_config)

train_loader = data_factory.train_loader
valid_loader = data_factory.valid_loader

train_iter = iter(train_loader)

print(f"train batches: {len(train_loader)}")
print(f"valid batches: {len(valid_loader)}")

.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


📦 Initializing datasets...
[Info] Parsed 42149 train items and 2132 test items from vctk
[Info] Parsed 149711 train items and 0 test items from libritts
[Info] Filtering items by duration (1.5s ~ 15.0s)...


Filtering train data: 100%|██████████| 191860/191860 [02:17<00:00, 1393.24it/s]


[Info] Filtering train data: filtered 41575 items. Remaining: 150285


Filtering test data: 100%|██████████| 2132/2132 [00:01<00:00, 1612.84it/s]


[Info] Filtering test data: filtered 768 items. Remaining: 1364
[Info] Data split complete: 148783 train, 1502 valid, 1364 test
train batches: 18590
valid batches: 188


In [2]:
def summarize_batch(batch, batch_idx: int | None = None):
    x_lengths = batch.text_lengths
    y_lengths = batch.spec_lengths

    x = x_lengths.detach().cpu()
    y = y_lengths.detach().cpu()

    ratio = y.float() / x.clamp_min(1).float()

    B = int(x.numel())
    T_text_max = int(x.max().item())
    T_mel_max = int(y.max().item())

    pair_area = B * T_mel_max * T_text_max

    text_pad_total = B * T_text_max
    text_real_total = int(x.sum().item())
    text_pad_ratio = 1.0 - (text_real_total / max(text_pad_total, 1))

    mel_pad_total = B * T_mel_max
    mel_real_total = int(y.sum().item())
    mel_pad_ratio = 1.0 - (mel_real_total / max(mel_pad_total, 1))

    per_item_pair = x * y

    title = f"[batch {batch_idx}]" if batch_idx is not None else "[batch]"

    print("=" * 120)
    print(title)
    print(f"B={B}")
    print(f"T_text_max={T_text_max}")
    print(f"T_mel_max={T_mel_max}")
    print(f"B*T_mel*T_text={pair_area:,}")
    print("-" * 120)

    print("x_lengths:", x.tolist())
    print("y_lengths:", y.tolist())
    print("ratio y/x:", [round(v, 3) for v in ratio.tolist()])

    print("-" * 120)
    print(
        "x stats | "
        + f"min={int(x.min())}, "
        + f"mean={float(x.float().mean()):.2f}, "
        + f"median={float(x.float().median()):.2f}, "
        + f"max={int(x.max())}"
    )
    print(
        "y stats | "
        + f"min={int(y.min())}, "
        + f"mean={float(y.float().mean()):.2f}, "
        + f"median={float(y.float().median()):.2f}, "
        + f"max={int(y.max())}"
    )
    print(
        "ratio stats | "
        + f"min={float(ratio.min()):.3f}, "
        + f"mean={float(ratio.mean()):.3f}, "
        + f"median={float(ratio.median()):.3f}, "
        + f"max={float(ratio.max()):.3f}"
    )
    print(
        "per-item pair x*y | "
        + f"min={int(per_item_pair.min())}, "
        + f"mean={float(per_item_pair.float().mean()):.1f}, "
        + f"max={int(per_item_pair.max())}"
    )

    print("-" * 120)
    print(f"text padding waste: {text_pad_ratio * 100:.2f}%")
    print(f"mel padding waste:  {mel_pad_ratio * 100:.2f}%")

    suspicious = []
    for i, (xl, yl, r) in enumerate(zip(x.tolist(), y.tolist(), ratio.tolist())):
        flags = []
        if yl < xl:
            flags.append("BAD: y_len < x_len")
        if r < 1.05:
            flags.append("low_ratio")
        if r > 10.0:
            flags.append("high_ratio")
        if flags:
            suspicious.append((i, xl, yl, round(r, 3), flags))

    if suspicious:
        print("-" * 120)
        print("⚠️ suspicious samples:")
        for item in suspicious:
            print(item)

    if hasattr(batch, "scripts"):
        print("-" * 120)
        print("texts:")
        for i, s in enumerate(batch.scripts):
            print(f"[{i}] x={int(x[i])}, y={int(y[i])}, ratio={float(ratio[i]):.3f} | {s[:200]}")

    print("=" * 120)

In [3]:
batch_idx = 0

In [4]:
batch = next(train_iter)

batch_idx += 1
summarize_batch(batch, batch_idx=batch_idx)

[batch 1]
B=8
T_text_max=143
T_mel_max=675
B*T_mel*T_text=772,200
------------------------------------------------------------------------------------------------------------------------
x_lengths: [133, 139, 131, 128, 134, 139, 132, 143]
y_lengths: [622, 675, 573, 576, 600, 620, 542, 650]
ratio y/x: [4.677, 4.856, 4.374, 4.5, 4.478, 4.46, 4.106, 4.545]
------------------------------------------------------------------------------------------------------------------------
x stats | min=128, mean=134.88, median=133.00, max=143
y stats | min=542, mean=607.25, median=600.00, max=675
ratio stats | min=4.106, mean=4.500, median=4.478, max=4.856
per-item pair x*y | min=71544, mean=82052.0, max=93825
------------------------------------------------------------------------------------------------------------------------
text padding waste: 5.68%
mel padding waste:  10.04%
------------------------------------------------------------------------------------------------------------------------
te